In [22]:
def MetageneAnalysis(inputFile,outputFile,mode="st",type="codon",cdslength=600,expression=75):
    # choose which mode to analysis
    if mode == "st":
        mylist = range(-50, 1501)
        # codon coorodinate
        region = list(range(-51,1501,3))
        codonPos = list(range(-17,501))
    elif mode == "sp":
        mylist = range(-1500, 51)
        # codon coorodinate
        region = list(range(-1501,51,3))
        codonPos = list(range(-500,18))
    else:
        print("pelase give the st/sp mode")

    # save info
    rangeDict = dict([i, 0] for i in mylist)
    countDict = dict([i, 0] for i in mylist)

    #########################################################
    # filter gene CDS length and expression higher thean threshold
    gene_infoDict = {}
    filtedGeneDict = {}
    
    # open file
    with open(inputFile,'r') as input:
        for line in input:
            fileds = line.split()

            # tags AAC1|-|51|977|930
            gene_name,_,cdsStart,cdsEnd,_ = fileds[0].split("|")
            pos = int(fileds[1])
            cdsLength = int(cdsEnd) - int(cdsStart) + 1
            density = float(fileds[3])

            # filter CDS > 400 nt gene
            if cdsLength > cdslength and cdsLength%3 == 0:
                # key
                key = ':'.join([gene_name,str(cdsLength - 90)])
                gene_infoDict.setdefault(key,0)
                if int(cdsStart)+90 <= pos <= int(cdsEnd):
                    gene_infoDict[key] += density
                else:
                    pass
            else:
                pass

    # filter CDS expression > 50
    for key,val in gene_infoDict.items():
        if val > expression:
            meanNorm = val/int(key.split(':')[1])
            filtedGeneDict[key] = meanNorm
        else:
            pass
    #########################################################
    # Meta-gene analysis from start codon

    # open file
    with open(inputFile,'r') as input:
        for line in input:
            fileds = line.split()

            # tags
            gene_name,_,cdsStart,cdsEnd,_ = fileds[0].split("|")
            pos = int(fileds[1])
            cdsLength = int(cdsEnd) - int(cdsStart) + 1
            density = float(fileds[3])
            id = ':'.join([gene_name,str(cdsLength - 90)])

            # calculate -50-1500 sum denisty
            if id in filtedGeneDict:
                # calculate relative to start/stop codon distance
                if mode == "st":
                    reldist = pos - int(cdsStart)
                elif mode == "sp":
                    reldist = pos - int(cdsEnd)
                else:
                    print("pelase give the st/sp mode")
                # sum up reads
                if mylist[0] <= reldist <= mylist[-1]:
                    # divide reads at one position by average number of reads per base for this gene
                    reads = density / filtedGeneDict[id]
                    rangeDict[reldist] += reads
                    # how often was this position counted in the calculation
                    countDict[reldist] += 1
                else:
                    pass
            else:
                pass
    
    #########################################################
    # output data
    # sort dict
    tupledlist1 = list(rangeDict.items())
    tupledlist1.sort()
    tupledlist2 = list(countDict.items())
    tupledlist2.sort()

    fullDict = {}
    zippedlist = zip(tupledlist1, tupledlist2)
    for elem in zippedlist:
        col0 = elem[0][0]       # list0 col0 = position (K)
        col1 = elem[0][1]       # list0 col1 = norm read number 
        col2 = elem[1][1]       # list1 col1 = how often was position counted

        #normalization2 by frequnecy
        if col2 == 0:
            fullDict[col0] = 0
        else:
            fullDict[col0] = col1 / col2 

    # calculate relative density(position denisty/mean_density)
    new_fullDict = {}
    meanDensity = sum(fullDict.values())/1551
    for key,val in fullDict.items():
        relDensity = val/meanDensity
        new_fullDict[key] = relDensity

    # condon position transform
    if type == "codon":
        if mode == "st":
            codonDict = {}
            for i in range(0,len(region)):
                codonRegion = range(region[i],region[i] + 3)
                count = 0
                codonDict.setdefault(codonPos[i],0)
                # sum up codon three position densitys
                for j in codonRegion:
                    if j in new_fullDict:
                        codonDict[codonPos[i]] += new_fullDict[j]
                        count += 1
                # codon mean density
                codonDict[codonPos[i]] = codonDict[codonPos[i]]/count
            finalDict = codonDict
        elif mode == "sp":
            codonDict = {}
            for i in range(0,len(region)):
                codonRegion = range(region[i],region[i] + 3)
                count = 0
                codonDict.setdefault(codonPos[i],0)
                # sum up codon three position densitys
                for j in codonRegion:
                    if j in new_fullDict:
                        codonDict[codonPos[i]] += new_fullDict[j]
                        count += 1
                # codon mean density
                codonDict[codonPos[i]] = codonDict[codonPos[i]]/count
            finalDict = codonDict
        else:
            finalDict = new_fullDict
    elif type == "nt":
        finalDict = new_fullDict
    else:
        print("pelase give the codon/nt mode")
                
    # Finish output
    tupledlist = list(finalDict.items())
    tupledlist.sort()

    # output
    outFileP = open(outputFile, 'w')
        
    for elem in tupledlist:
        outFileP.write('\t'.join([str(elem[0]),str(elem[1])]) + '\n')
    outFileP.close()

In [4]:
import os

os.mkdir('./4.metagene-data/')

In [23]:
sample = ['ssb1-inter-rep1.map.density.txt','ssb1-inter-rep2.map.density.txt','ssb2-inter-rep1.map.density.txt','ssb2-inter-rep2.map.density.txt',
          'ssb1-trans-rep1.map.density.txt','ssb1-trans-rep2.map.density.txt','ssb2-trans-rep1.map.density.txt','ssb2-trans-rep2.map.density.txt']

# run
for i in range(0,8):
    MetageneAnalysis(''.join(['3.ribo-density-data/',sample[i]]),''.join(['4.metagene-data/',sample[i],'.metegene2STCodon.txt']),
                    mode="st",type="codon",cdslength=400,expression=50)